# ODI to Databricks Migration: TARGET_FILE_NAME

**Conversion Timestamp:** 2024-07-30 00:00:00 (Placeholder)

**Description:** This notebook migrates an ODI session to perform an INSERT into `TRG_DEP` table from `DEPARTMENTS`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "F", "ETL Job Type (F/I)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id,
  CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid,
  CAST(${ODI_SESS_NO} AS BIGINT) AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Staging Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}, {20}, {30}: These are placeholder tasks in the original ODI script.
-- The actual INSERT statement follows here.

## Flow Table

In [ ]:
%sql
-- No specific flow table operations for this simple INSERT, proceeding to target load.

## Error / Check Tables

In [ ]:
%sql
-- No error table DDL or DML in original script. Example placeholder for E$ tables:
-- CREATE TABLE IF NOT EXISTS workspace.hr.e_trg_dep (
--   ODI_SESS_NO BIGINT,
--   ERROR_MESSAGE STRING,
--   DEPARTMENT_ID BIGINT
-- ) USING DELTA;
-- DELETE FROM workspace.hr.e_trg_dep WHERE ODI_SESS_NO = ${ODI_SESS_NO};

## MERGE into Target

In [ ]:
%sql
-- SCEN_TASK_NO {30} continued: Perform INSERT into TRG_DEP
INSERT INTO workspace.hr.trg_dep
(
  DEPARTMENT_ID,
  DEPARTMENT_NAME,
  MANAGER_ID,
  LOCATION_ID
)
SELECT
  DEPARTMENTS.DEPARTMENT_ID,
  DEPARTMENTS.DEPARTMENT_NAME,
  DEPARTMENTS.MANAGER_ID,
  DEPARTMENTS.LOCATION_ID
FROM
  workspace.hr.departments AS DEPARTMENTS;

## Cleanup

In [ ]:
%sql
-- No staging or flow tables were explicitly created in this specific script portion to drop.

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS final_record_count FROM workspace.hr.trg_dep;

In [ ]:
%sql
SELECT *
FROM workspace.hr.trg_dep
ORDER BY DEPARTMENT_ID DESC
LIMIT 10;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema and table names have been converted to lowercase and prefixed with `workspace.`. For example, `HR.TRG_DEP` is now `workspace.hr.trg_dep` and `HR.DEPARTMENTS` is `workspace.hr.departments`.
2.  **Oracle Hints:** The Oracle hint `/*+ APPEND PARALLEL */` has been removed as it is not applicable in Databricks Delta Lake.
3.  **SCEN_TASK_NO:** Initial `SCEN_TASK_NO` lines without explicit SQL statements (e.g., {10}, {20}, {30}) are treated as markers and commented in the relevant SQL cell.
4.  **Target Table DDL:** The original script only contained an `INSERT` statement. Ensure that the target table `workspace.hr.trg_dep` is created with the appropriate Spark SQL data types (e.g., `BIGINT` for Oracle `NUMBER(p,0)`, `STRING` for `VARCHAR2`, `TIMESTAMP` for `DATE`) before running this notebook.
5.  **Parameter Views:** Temporary views for ETL parameters (`v_etl_parameters`) have been added, though the specific SQL from the source does not directly use them. This follows the standard notebook structure for Databricks ETL processes.
